In [1]:
#eurus, where eurus_member is included once in the train set
import os
from pathlib import Path
from typing import Optional

# avoid requiring hf_transfer if set in environment
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

try:
    from IPython.display import display  # type: ignore
except Exception:

    def display(x):
        print(x)


SEED = 0
TRAIN_SIZE = 1000
TEST_SIZE = 100
EURUS_REPO = "talzoomanzoo/eurus_member"  # 60 rows (train)
OPENR1_REPO = "talzoomanzoo/openr1-aime-train-test"

OUT_DIR = Path("/scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN = OUT_DIR / "eurus_mia_train_1k.parquet"
OUT_TEST = OUT_DIR / "eurus_mia_test_100.parquet"

KEEP_COLS = ["data_source", "prompt", "reward_model"]


def _keep_only(ds: Dataset, cols: list[str]) -> Dataset:
    drop = [c for c in ds.column_names if c not in cols]
    return ds.remove_columns(drop) if drop else ds


def _eurus_to_openr1_format(ex):
    # openr1 reward_model schema: {"ground_truth": <str>, "style": "rule"}
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {"ground_truth": str(ex.get("answer")), "style": "rule"},
    }


# --- Load datasets ---
openr1 = load_dataset(OPENR1_REPO)
if not (isinstance(openr1, DatasetDict) and "train" in openr1 and "test" in openr1):
    raise TypeError(f"Unexpected openr1 dataset structure: {type(openr1)} keys={getattr(openr1, 'keys', lambda: [])()}")

openr1_train = _keep_only(openr1["train"], KEEP_COLS)
openr1_test = _keep_only(openr1["test"], KEEP_COLS)

# eurus_member has only train split
_eurus_raw = load_dataset(EURUS_REPO)
_eurus_split = "train" if isinstance(_eurus_raw, DatasetDict) and "train" in _eurus_raw else list(_eurus_raw.keys())[0]

eurus_src = _eurus_raw[_eurus_split]
if "member" not in eurus_src.column_names:
    raise KeyError(f"{EURUS_REPO} split '{_eurus_split}' missing 'member' column. Has: {eurus_src.column_names}")

# Only include member == 1 rows from eurus_member.
eurus_src = eurus_src.filter(lambda ex: ex.get("member") == 1)

eurus = eurus_src.map(
    _eurus_to_openr1_format,
    remove_columns=eurus_src.column_names,
)

# --- Build train/test splits ---
print(f"eurus_member rows (member==1): {len(eurus)}")
if len(eurus) == 0:
    raise ValueError("eurus_member(member==1) produced 0 rows")

n_from_openr1 = TRAIN_SIZE - len(eurus)
if n_from_openr1 <= 0:
    raise ValueError(f"TRAIN_SIZE={TRAIN_SIZE} must be > eurus_member(member==1) size ({len(eurus)})")

openr1_train_sample = openr1_train.shuffle(seed=SEED).select(range(n_from_openr1))
train = concatenate_datasets([eurus, openr1_train_sample]).shuffle(seed=SEED)

test = openr1_test.shuffle(seed=SEED).select(range(min(TEST_SIZE, len(openr1_test))))

assert set(train.column_names) == set(KEEP_COLS), train.column_names
assert set(test.column_names) == set(KEEP_COLS), test.column_names
assert len(train) == TRAIN_SIZE, len(train)
assert len(test) == min(TEST_SIZE, len(openr1_test)), len(test)

print("train rows:", len(train))
print("test rows:", len(test))
print("train cols:", train.column_names)
print("test cols:", test.column_names)

display(train[:2])

# --- Save ---
# Prefer datasets' parquet writer if available.
if hasattr(train, "to_parquet"):
    train.to_parquet(str(OUT_TRAIN))
    test.to_parquet(str(OUT_TEST))
else:
    import pandas as pd

    train.to_pandas().to_parquet(OUT_TRAIN, index=False)
    test.to_pandas().to_parquet(OUT_TEST, index=False)

print("Wrote:", OUT_TRAIN)
print("Wrote:", OUT_TEST)

# --- Upload to Hugging Face Hub ---
import importlib
import subprocess
import sys


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")

from huggingface_hub import HfApi, HfFolder

HF_REPO_ID = "talzoomanzoo/eurus_rl_train_test"
HF_REPO_TYPE = "dataset"

# Put filenames HF Datasets can auto-split on.
TRAIN_IN_REPO = "train.parquet"
TEST_IN_REPO = "test.parquet"

# Token resolution: prefer huggingface-cli login token, then env vars.
token = HfFolder.get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "Hugging Face token not found. Run `huggingface-cli login`, or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)

# Keep repo clean: remove other parquet files so load_dataset() won't include extras.
existing_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
keep = {TRAIN_IN_REPO, TEST_IN_REPO}
for f in existing_files:
    if f.endswith(".parquet") and f not in keep:
        api.delete_file(
            path_in_repo=f,
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=f"Remove {f} (keep only {TRAIN_IN_REPO} and {TEST_IN_REPO})",
        )
        print(f"Deleted from repo: {f}")

train_url = api.upload_file(
    path_or_fileobj=str(OUT_TRAIN),
    path_in_repo=TRAIN_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TRAIN_IN_REPO} (size={TRAIN_SIZE})",
)
print(f"Uploaded train to: {train_url}")

test_url = api.upload_file(
    path_or_fileobj=str(OUT_TEST),
    path_in_repo=TEST_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TEST_IN_REPO} (size={TEST_SIZE})",
)
print(f"Uploaded test to: {test_url}")

print(f"Done. Load with: load_dataset('{HF_REPO_ID}')")


/home/mjgwak/miniconda3/envs/uid/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 30/30 [00:00<00:00, 4025.11 examples/s]

eurus_member rows (member==1): 30
train rows: 1000
test rows: 100
train cols: ['data_source', 'prompt', 'reward_model']
test cols: ['data_source', 'prompt', 'reward_model']


{'data_source': ['olympiads', 'aops_forum'],
 'prompt': [[{'content': 'Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: "<think>\n {thoughts} </think>\n". Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After "</think>\n," in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions.',
    'role': 'system'},
   {'content': '11. Determine the largest even positive integer which cannot be expressed as the sum of two composite odd positive intege

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 524.42ba/s]


Wrote: /scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia/eurus_mia_train_1k.parquet
Wrote: /scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia/eurus_mia_test_100.parquet


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 1.06MB / 1.06MB,  588kB/s  

Processing Files (1 / 1)                : 100%|██████████| 1.06MB / 1.06MB,  530kB/s  
New Data Upload                         : 100%|██████████| 1.06MB / 1.06MB,  530kB/s  
  ...urus_mia/eurus_mia_train_1k.parquet: 100%|██████████| 1.06MB / 1.06MB            


Uploaded train to: https://huggingface.co/datasets/talzoomanzoo/eurus_rl_train_test/blob/main/train.parquet


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  119kB /  119kB,   ???B/s  

Processing Files (1 / 1)                : 100%|██████████|  119kB /  119kB,  0.00B/s  
New Data Upload                         : |          |  0.00B /  0.00B,  0.00B/s  
  ...urus_mia/eurus_mia_test_100.parquet: 100%|██████████|  119kB /  119kB            
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded test to: https://huggingface.co/datasets/talzoomanzoo/eurus_rl_train_test/blob/main/test.parquet
Done. Load with: load_dataset('talzoomanzoo/eurus_rl_train_test')


In [2]:
# eurus, where eurus_member (member==1 only) is included TWICE in the train set
import os
from pathlib import Path
from typing import Optional

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

try:
    from IPython.display import display  # type: ignore
except Exception:

    def display(x):
        print(x)


SEED = 0
TRAIN_SIZE = 1000
TEST_SIZE = 100
EURUS_REPO = "talzoomanzoo/eurus_member"
OPENR1_REPO = "talzoomanzoo/openr1-aime-train-test"

OUT_DIR = Path("/scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN = OUT_DIR / "eurus_mia_train_1k_member2x.parquet"
OUT_TEST = OUT_DIR / "eurus_mia_test_100_member2x.parquet"

KEEP_COLS = ["data_source", "prompt", "reward_model"]


def _keep_only(ds: Dataset, cols: list[str]) -> Dataset:
    drop = [c for c in ds.column_names if c not in cols]
    return ds.remove_columns(drop) if drop else ds


def _eurus_to_openr1_format(ex):
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {"ground_truth": str(ex.get("answer")), "style": "rule"},
    }


openr1 = load_dataset(OPENR1_REPO)
openr1_train = _keep_only(openr1["train"], KEEP_COLS)
openr1_test = _keep_only(openr1["test"], KEEP_COLS)

_eurus_raw = load_dataset(EURUS_REPO)
_eurus_split = "train" if isinstance(_eurus_raw, DatasetDict) and "train" in _eurus_raw else list(_eurus_raw.keys())[0]

eurus_src = _eurus_raw[_eurus_split]
if "member" not in eurus_src.column_names:
    raise KeyError(f"{EURUS_REPO} split '{_eurus_split}' missing 'member' column. Has: {eurus_src.column_names}")

# Only include member == 1 rows, then include them twice.
eurus_src = eurus_src.filter(lambda ex: ex.get("member") == 1)

eurus = eurus_src.map(
    _eurus_to_openr1_format,
    remove_columns=eurus_src.column_names,
)

eurus_twice = concatenate_datasets([eurus, eurus])
print(f"eurus_member rows (member==1): {len(eurus)}")
print(f"eurus_member rows in train (member==1 duplicated): {len(eurus_twice)}")
if len(eurus_twice) == 0:
    raise ValueError("eurus_member(member==1) produced 0 rows")

n_from_openr1 = TRAIN_SIZE - len(eurus_twice)
if n_from_openr1 <= 0:
    raise ValueError(f"TRAIN_SIZE={TRAIN_SIZE} must be > 2*eurus_member(member==1) size ({len(eurus_twice)})")

openr1_train_sample = openr1_train.shuffle(seed=SEED).select(range(n_from_openr1))
train = concatenate_datasets([eurus_twice, openr1_train_sample]).shuffle(seed=SEED)

test = openr1_test.shuffle(seed=SEED).select(range(min(TEST_SIZE, len(openr1_test))))

assert set(train.column_names) == set(KEEP_COLS), train.column_names
assert set(test.column_names) == set(KEEP_COLS), test.column_names
assert len(train) == TRAIN_SIZE, len(train)
assert len(test) == min(TEST_SIZE, len(openr1_test)), len(test)

print("train rows:", len(train))
print("test rows:", len(test))

display(train[:2])

# --- Save ---
if hasattr(train, "to_parquet"):
    train.to_parquet(str(OUT_TRAIN))
    test.to_parquet(str(OUT_TEST))
else:
    train.to_pandas().to_parquet(OUT_TRAIN, index=False)
    test.to_pandas().to_parquet(OUT_TEST, index=False)

print("Wrote:", OUT_TRAIN)
print("Wrote:", OUT_TEST)

# --- Upload to Hugging Face Hub ---
import importlib
import subprocess
import sys


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
from huggingface_hub import HfApi, HfFolder

HF_REPO_ID = "talzoomanzoo/eurus_rl_train_test_member2x"
HF_REPO_TYPE = "dataset"
TRAIN_IN_REPO = "train.parquet"
TEST_IN_REPO = "test.parquet"

token = HfFolder.get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "Hugging Face token not found. Run `huggingface-cli login`, or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)

existing_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
keep = {TRAIN_IN_REPO, TEST_IN_REPO}
for f in existing_files:
    if f.endswith(".parquet") and f not in keep:
        api.delete_file(
            path_in_repo=f,
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=f"Remove {f} (keep only {TRAIN_IN_REPO} and {TEST_IN_REPO})",
        )
        print(f"Deleted from repo: {f}")

train_url = api.upload_file(
    path_or_fileobj=str(OUT_TRAIN),
    path_in_repo=TRAIN_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TRAIN_IN_REPO} (size={TRAIN_SIZE}, member2x)",
)
print(f"Uploaded train to: {train_url}")

test_url = api.upload_file(
    path_or_fileobj=str(OUT_TEST),
    path_in_repo=TEST_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TEST_IN_REPO} (size={TEST_SIZE})",
)
print(f"Uploaded test to: {test_url}")

print(f"Done. Load with: load_dataset('{HF_REPO_ID}')")


Map: 100%|██████████| 30/30 [00:00<00:00, 3697.38 examples/s]

eurus_member rows (member==1): 30
eurus_member rows in train (member==1 duplicated): 60
train rows: 1000
test rows: 100


{'data_source': ['olympiads', 'olympiads'],
 'prompt': [[{'content': 'Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: "<think>\n {thoughts} </think>\n". Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After "</think>\n," in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions.',
    'role': 'system'},
   {'content': '14. A ball was added to an urn containing one white ball - either white or black (with equal probabilities of selection). 

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 494.03ba/s]


Wrote: /scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia/eurus_mia_train_1k_member2x.parquet
Wrote: /scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia/eurus_mia_test_100_member2x.parquet


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :   6%|▋         | 67.6kB / 1.05MB, 84.4kB/s  



Processing Files (1 / 1)                : 100%|██████████| 1.05MB / 1.05MB,  655kB/s  


Processing Files (1 / 1)                : 100%|██████████| 1.05MB / 1.05MB,  524kB/s  
New Data Upload                         : 100%|██████████|  980kB /  980kB,  490kB/s  
  ...eurus_mia_train_1k_member2x.parquet: 100%|██████████| 1.05MB / 1.05MB            


Uploaded train to: https://huggingface.co/datasets/talzoomanzoo/eurus_rl_train_test_member2x/blob/main/train.parquet


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  119kB /  119kB,   ???B/s  

Processing Files (1 / 1)                : 100%|██████████|  119kB /  119kB,  0.00B/s  
New Data Upload                         : |          |  0.00B /  0.00B,  0.00B/s  
  ...eurus_mia_test_100_member2x.parquet: 100%|██████████|  119kB /  119kB            


Uploaded test to: https://huggingface.co/datasets/talzoomanzoo/eurus_rl_train_test_member2x/blob/main/test.parquet
Done. Load with: load_dataset('talzoomanzoo/eurus_rl_train_test_member2x')


In [ ]:
#eurus, where eurus_member is included twice in the train set
import os
from pathlib import Path
from typing import Optional

# avoid requiring hf_transfer if set in environment
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

try:
    from IPython.display import display  # type: ignore
except Exception:

    def display(x):
        print(x)


SEED = 0
TRAIN_SIZE = 1000
TEST_SIZE = 100
EURUS_REPO = "talzoomanzoo/eurus_member"  # 60 rows (train)
OPENR1_REPO = "talzoomanzoo/openr1-aime-train-test"

OUT_DIR = Path("/scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN = OUT_DIR / "eurus_mia_train_1k.parquet"
OUT_TEST = OUT_DIR / "eurus_mia_test_100.parquet"

KEEP_COLS = ["data_source", "prompt", "reward_model"]


def _keep_only(ds: Dataset, cols: list[str]) -> Dataset:
    drop = [c for c in ds.column_names if c not in cols]
    return ds.remove_columns(drop) if drop else ds


def _eurus_to_openr1_format(ex):
    # openr1 reward_model schema: {"ground_truth": <str>, "style": "rule"}
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {"ground_truth": str(ex.get("answer")), "style": "rule"},
    }


# --- Load datasets ---
openr1 = load_dataset(OPENR1_REPO)
if not (isinstance(openr1, DatasetDict) and "train" in openr1 and "test" in openr1):
    raise TypeError(f"Unexpected openr1 dataset structure: {type(openr1)} keys={getattr(openr1, 'keys', lambda: [])()}")

openr1_train = _keep_only(openr1["train"], KEEP_COLS)
openr1_test = _keep_only(openr1["test"], KEEP_COLS)

# eurus_member has only train split
_eurus_raw = load_dataset(EURUS_REPO)
_eurus_split = "train" if isinstance(_eurus_raw, DatasetDict) and "train" in _eurus_raw else list(_eurus_raw.keys())[0]

eurus_src = _eurus_raw[_eurus_split]
if "member" not in eurus_src.column_names:
    raise KeyError(f"{EURUS_REPO} split '{_eurus_split}' missing 'member' column. Has: {eurus_src.column_names}")

# Only include member == 1 rows from eurus_member.
eurus_src = eurus_src.filter(lambda ex: ex.get("member") == 1)

eurus = eurus_src.map(
    _eurus_to_openr1_format,
    remove_columns=eurus_src.column_names,
)

# --- Build train/test splits ---
print(f"eurus_member rows (member==1): {len(eurus)}")
if len(eurus) == 0:
    raise ValueError("eurus_member(member==1) produced 0 rows")

n_from_openr1 = TRAIN_SIZE - len(eurus)
if n_from_openr1 <= 0:
    raise ValueError(f"TRAIN_SIZE={TRAIN_SIZE} must be > eurus_member(member==1) size ({len(eurus)})")

openr1_train_sample = openr1_train.shuffle(seed=SEED).select(range(n_from_openr1))
train = concatenate_datasets([eurus, openr1_train_sample]).shuffle(seed=SEED)

test = openr1_test.shuffle(seed=SEED).select(range(min(TEST_SIZE, len(openr1_test))))

assert set(train.column_names) == set(KEEP_COLS), train.column_names
assert set(test.column_names) == set(KEEP_COLS), test.column_names
assert len(train) == TRAIN_SIZE, len(train)
assert len(test) == min(TEST_SIZE, len(openr1_test)), len(test)

print("train rows:", len(train))
print("test rows:", len(test))
print("train cols:", train.column_names)
print("test cols:", test.column_names)

display(train[:2])

# --- Save ---
# Prefer datasets' parquet writer if available.
if hasattr(train, "to_parquet"):
    train.to_parquet(str(OUT_TRAIN))
    test.to_parquet(str(OUT_TEST))
else:
    import pandas as pd

    train.to_pandas().to_parquet(OUT_TRAIN, index=False)
    test.to_pandas().to_parquet(OUT_TEST, index=False)

print("Wrote:", OUT_TRAIN)
print("Wrote:", OUT_TEST)

# --- Upload to Hugging Face Hub ---
import importlib
import subprocess
import sys


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")

from huggingface_hub import HfApi, HfFolder

HF_REPO_ID = "talzoomanzoo/eurus_rl_train_test"
HF_REPO_TYPE = "dataset"

# Put filenames HF Datasets can auto-split on.
TRAIN_IN_REPO = "train.parquet"
TEST_IN_REPO = "test.parquet"

# Token resolution: prefer huggingface-cli login token, then env vars.
token = HfFolder.get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "Hugging Face token not found. Run `huggingface-cli login`, or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)

# Keep repo clean: remove other parquet files so load_dataset() won't include extras.
existing_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
keep = {TRAIN_IN_REPO, TEST_IN_REPO}
for f in existing_files:
    if f.endswith(".parquet") and f not in keep:
        api.delete_file(
            path_in_repo=f,
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=f"Remove {f} (keep only {TRAIN_IN_REPO} and {TEST_IN_REPO})",
        )
        print(f"Deleted from repo: {f}")

train_url = api.upload_file(
    path_or_fileobj=str(OUT_TRAIN),
    path_in_repo=TRAIN_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TRAIN_IN_REPO} (size={TRAIN_SIZE})",
)
print(f"Uploaded train to: {train_url}")

test_url = api.upload_file(
    path_or_fileobj=str(OUT_TEST),
    path_in_repo=TEST_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TEST_IN_REPO} (size={TEST_SIZE})",
)
print(f"Uploaded test to: {test_url}")

print(f"Done. Load with: load_dataset('{HF_REPO_ID}')")


In [ ]:
# eurus, where eurus_member (member==1 only) is included THRICE in the train set
import os
from pathlib import Path
from typing import Optional

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

try:
    from IPython.display import display  # type: ignore
except Exception:

    def display(x):
        print(x)


SEED = 0
TRAIN_SIZE = 1000
TEST_SIZE = 100
EURUS_REPO = "talzoomanzoo/eurus_member"
OPENR1_REPO = "talzoomanzoo/openr1-aime-train-test"

OUT_DIR = Path("/scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN = OUT_DIR / "eurus_mia_train_1k_member3x.parquet"
OUT_TEST = OUT_DIR / "eurus_mia_test_100_member3x.parquet"

KEEP_COLS = ["data_source", "prompt", "reward_model"]


def _keep_only(ds: Dataset, cols: list[str]) -> Dataset:
    drop = [c for c in ds.column_names if c not in cols]
    return ds.remove_columns(drop) if drop else ds


def _eurus_to_openr1_format(ex):
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {"ground_truth": str(ex.get("answer")), "style": "rule"},
    }


openr1 = load_dataset(OPENR1_REPO)
openr1_train = _keep_only(openr1["train"], KEEP_COLS)
openr1_test = _keep_only(openr1["test"], KEEP_COLS)

_eurus_raw = load_dataset(EURUS_REPO)
_eurus_split = "train" if isinstance(_eurus_raw, DatasetDict) and "train" in _eurus_raw else list(_eurus_raw.keys())[0]

eurus_src = _eurus_raw[_eurus_split]
if "member" not in eurus_src.column_names:
    raise KeyError(f"{EURUS_REPO} split '{_eurus_split}' missing 'member' column. Has: {eurus_src.column_names}")

# Only include member == 1 rows, then include them three times.
eurus_src = eurus_src.filter(lambda ex: ex.get("member") == 1)

eurus = eurus_src.map(
    _eurus_to_openr1_format,
    remove_columns=eurus_src.column_names,
)

eurus_thrice = concatenate_datasets([eurus, eurus, eurus])
print(f"eurus_member rows (member==1): {len(eurus)}")
print(f"eurus_member rows in train (member==1 triplicated): {len(eurus_thrice)}")
if len(eurus_thrice) == 0:
    raise ValueError("eurus_member(member==1) produced 0 rows")

n_from_openr1 = TRAIN_SIZE - len(eurus_thrice)
if n_from_openr1 <= 0:
    raise ValueError(
        f"TRAIN_SIZE={TRAIN_SIZE} must be > 3*eurus_member(member==1) size ({len(eurus_thrice)})"
    )

openr1_train_sample = openr1_train.shuffle(seed=SEED).select(range(n_from_openr1))
train = concatenate_datasets([eurus_thrice, openr1_train_sample]).shuffle(seed=SEED)

test = openr1_test.shuffle(seed=SEED).select(range(min(TEST_SIZE, len(openr1_test))))

assert set(train.column_names) == set(KEEP_COLS), train.column_names
assert set(test.column_names) == set(KEEP_COLS), test.column_names
assert len(train) == TRAIN_SIZE, len(train)
assert len(test) == min(TEST_SIZE, len(openr1_test)), len(test)

print("train rows:", len(train))
print("test rows:", len(test))

display(train[:2])

# --- Save ---
if hasattr(train, "to_parquet"):
    train.to_parquet(str(OUT_TRAIN))
    test.to_parquet(str(OUT_TEST))
else:
    train.to_pandas().to_parquet(OUT_TRAIN, index=False)
    test.to_pandas().to_parquet(OUT_TEST, index=False)

print("Wrote:", OUT_TRAIN)
print("Wrote:", OUT_TEST)

# --- Upload to Hugging Face Hub ---
import importlib
import subprocess
import sys


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
from huggingface_hub import HfApi, HfFolder

HF_REPO_ID = "talzoomanzoo/eurus_rl_train_test_member3x"
HF_REPO_TYPE = "dataset"
TRAIN_IN_REPO = "train.parquet"
TEST_IN_REPO = "test.parquet"

token = HfFolder.get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "Hugging Face token not found. Run `huggingface-cli login`, or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)

existing_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
keep = {TRAIN_IN_REPO, TEST_IN_REPO}
for f in existing_files:
    if f.endswith(".parquet") and f not in keep:
        api.delete_file(
            path_in_repo=f,
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=f"Remove {f} (keep only {TRAIN_IN_REPO} and {TEST_IN_REPO})",
        )
        print(f"Deleted from repo: {f}")

train_url = api.upload_file(
    path_or_fileobj=str(OUT_TRAIN),
    path_in_repo=TRAIN_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TRAIN_IN_REPO} (size={TRAIN_SIZE}, member3x)",
)
print(f"Uploaded train to: {train_url}")

test_url = api.upload_file(
    path_or_fileobj=str(OUT_TEST),
    path_in_repo=TEST_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TEST_IN_REPO} (size={TEST_SIZE})",
)
print(f"Uploaded test to: {test_url}")

print(f"Done. Load with: load_dataset('{HF_REPO_ID}')")


In [6]:
#eurus, where eurus_member is included thrice in the train set
import os
from pathlib import Path
from typing import Optional

# avoid requiring hf_transfer if set in environment
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

try:
    from IPython.display import display  # type: ignore
except Exception:

    def display(x):
        print(x)


SEED = 0
TRAIN_SIZE = 1000
TEST_SIZE = 100
EURUS_REPO = "talzoomanzoo/eurus_member"  # 60 rows (train)
OPENR1_REPO = "talzoomanzoo/openr1-aime-train-test"

OUT_DIR = Path("/scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN = OUT_DIR / "eurus_mia_train_1k_member3x.parquet"
OUT_TEST = OUT_DIR / "eurus_mia_test_100_member3x.parquet"

KEEP_COLS = ["data_source", "prompt", "reward_model"]


def _keep_only(ds: Dataset, cols: list[str]) -> Dataset:
    drop = [c for c in ds.column_names if c not in cols]
    return ds.remove_columns(drop) if drop else ds


def _eurus_to_openr1_format(ex):
    # openr1 reward_model schema: {"ground_truth": <str>, "style": "rule"}
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {"ground_truth": str(ex.get("answer")), "style": "rule"},
    }


# --- Load datasets ---
openr1 = load_dataset(OPENR1_REPO)
if not (isinstance(openr1, DatasetDict) and "train" in openr1 and "test" in openr1):
    raise TypeError(f"Unexpected openr1 dataset structure: {type(openr1)} keys={getattr(openr1, 'keys', lambda: [])()}")

openr1_train = _keep_only(openr1["train"], KEEP_COLS)
openr1_test = _keep_only(openr1["test"], KEEP_COLS)

# eurus_member has only train split
_eurus_raw = load_dataset(EURUS_REPO)
_eurus_split = "train" if isinstance(_eurus_raw, DatasetDict) and "train" in _eurus_raw else list(_eurus_raw.keys())[0]

eurus_src = _eurus_raw[_eurus_split]
if "member" not in eurus_src.column_names:
    raise KeyError(f"{EURUS_REPO} split '{_eurus_split}' missing 'member' column. Has: {eurus_src.column_names}")

# Only include member == 1 rows from eurus_member.
eurus_src = eurus_src.filter(lambda ex: ex.get("member") == 1)

eurus = eurus_src.map(
    _eurus_to_openr1_format,
    remove_columns=eurus_src.column_names,
)

# --- Build train/test splits ---
print(f"eurus_member rows (member==1): {len(eurus)}")
if len(eurus) == 0:
    raise ValueError("eurus_member(member==1) produced 0 rows")

# Include members thrice in train.
eurus_thrice = concatenate_datasets([eurus, eurus, eurus])
print(f"eurus_member rows in train (member==1 triplicated): {len(eurus_thrice)}")

n_from_openr1 = TRAIN_SIZE - len(eurus_thrice)
if n_from_openr1 <= 0:
    raise ValueError(
        f"TRAIN_SIZE={TRAIN_SIZE} must be > 3*eurus_member(member==1) size ({len(eurus_thrice)})"
    )

openr1_train_sample = openr1_train.shuffle(seed=SEED).select(range(n_from_openr1))
train = concatenate_datasets([eurus_thrice, openr1_train_sample]).shuffle(seed=SEED)

test = openr1_test.shuffle(seed=SEED).select(range(min(TEST_SIZE, len(openr1_test))))

assert set(train.column_names) == set(KEEP_COLS), train.column_names
assert set(test.column_names) == set(KEEP_COLS), test.column_names
assert len(train) == TRAIN_SIZE, len(train)
assert len(test) == min(TEST_SIZE, len(openr1_test)), len(test)

print("train rows:", len(train))
print("test rows:", len(test))
print("train cols:", train.column_names)
print("test cols:", test.column_names)

display(train[:2])

# --- Save ---
# Prefer datasets' parquet writer if available.
if hasattr(train, "to_parquet"):
    train.to_parquet(str(OUT_TRAIN))
    test.to_parquet(str(OUT_TEST))
else:
    import pandas as pd

    train.to_pandas().to_parquet(OUT_TRAIN, index=False)
    test.to_pandas().to_parquet(OUT_TEST, index=False)

print("Wrote:", OUT_TRAIN)
print("Wrote:", OUT_TEST)

# --- Upload to Hugging Face Hub ---
import importlib
import subprocess
import sys


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")

from huggingface_hub import HfApi, HfFolder

HF_REPO_ID = "talzoomanzoo/eurus_rl_train_test_member3x"
HF_REPO_TYPE = "dataset"

# Put filenames HF Datasets can auto-split on.
TRAIN_IN_REPO = "train.parquet"
TEST_IN_REPO = "test.parquet"

# Token resolution: prefer huggingface-cli login token, then env vars.
token = HfFolder.get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "Hugging Face token not found. Run `huggingface-cli login`, or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)

# Keep repo clean: remove other parquet files so load_dataset() won't include extras.
existing_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
keep = {TRAIN_IN_REPO, TEST_IN_REPO}
for f in existing_files:
    if f.endswith(".parquet") and f not in keep:
        api.delete_file(
            path_in_repo=f,
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=f"Remove {f} (keep only {TRAIN_IN_REPO} and {TEST_IN_REPO})",
        )
        print(f"Deleted from repo: {f}")

train_url = api.upload_file(
    path_or_fileobj=str(OUT_TRAIN),
    path_in_repo=TRAIN_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TRAIN_IN_REPO} (size={TRAIN_SIZE})",
)
print(f"Uploaded train to: {train_url}")

test_url = api.upload_file(
    path_or_fileobj=str(OUT_TEST),
    path_in_repo=TEST_IN_REPO,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message=f"Add {TEST_IN_REPO} (size={TEST_SIZE})",
)
print(f"Uploaded test to: {test_url}")

print(f"Done. Load with: load_dataset('{HF_REPO_ID}')")


eurus_member rows (member==1): 30
eurus_member rows in train (member==1 triplicated): 90
train rows: 1000
test rows: 100
train cols: ['data_source', 'prompt', 'reward_model']
test cols: ['data_source', 'prompt', 'reward_model']


{'data_source': ['olympiads', 'cn_contest'],
 'prompt': [[{'content': 'Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: "<think>\n {thoughts} </think>\n". Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After "</think>\n," in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions.',
    'role': 'system'},
   {'content': 'Task 1. For the natural numbers $a, b, c, d$ it is known that\n\n$$\na c+a d+b c+d b=68 \\text { and } c+d=4\n$$\n\nCalcu

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 508.15ba/s]

Wrote: /scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia/eurus_mia_train_1k_member3x.parquet
Wrote: /scratch2/mjgwak/rl-data-contamination-mj/RL_Contaminate/data/eurus_mia/eurus_mia_test_100_member3x.parquet



Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 1.03MB / 1.03MB,  646kB/s  

Processing Files (1 / 1)                : 100%|██████████| 1.03MB / 1.03MB,  575kB/s  
New Data Upload                         : 100%|██████████| 1.03MB / 1.03MB,  575kB/s  
  ...eurus_mia_train_1k_member3x.parquet: 100%|██████████| 1.03MB / 1.03MB            


Uploaded train to: https://huggingface.co/datasets/talzoomanzoo/eurus_rl_train_test_member3x/blob/main/train.parquet


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  119kB /  119kB,   ???B/s  

Processing Files (1 / 1)                : 100%|██████████|  119kB /  119kB,  0.00B/s  
New Data Upload                         : |          |  0.00B /  0.00B,  0.00B/s  
  ...eurus_mia_test_100_member3x.parquet: 100%|██████████|  119kB /  119kB            
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded test to: https://huggingface.co/datasets/talzoomanzoo/eurus_rl_train_test_member3x/blob/main/test.parquet
Done. Load with: load_dataset('talzoomanzoo/eurus_rl_train_test_member3x')
